In [1]:
import pandas as pd

questions_df = pd.read_csv("../datasets/stackoverflow_dataset/Questions.csv", encoding='latin1')
answers_df = pd.read_csv("../datasets/stackoverflow_dataset/Answers.csv", encoding='latin1')
tags_df = pd.read_csv("../datasets/stackoverflow_dataset/Tags.csv", encoding='latin1')

In [8]:
tags_df.head()

,Id,Tag
0,469,python
1,469,osx
2,469,fonts
3,469,photoshop
4,502,python


In [6]:
print(questions_df.columns, "\n", answers_df.columns, "\n", tags_df.columns)

Index(['Id', 'OwnerUserId', 'CreationDate', 'Score', 'Title', 'Body'], dtype='object') 
 Index(['Id', 'OwnerUserId', 'CreationDate', 'ParentId', 'Score', 'Body'], dtype='object') 
 Index(['Id', 'Tag'], dtype='object')


In [21]:
from pprint import pprint
from tqdm import tqdm
formatted_data_dict = {}

for i in tqdm(range(len(questions_df))):
    
    # Extracting question data
    question_row = questions_df.iloc[i]
    question_id = question_row["Id"]
    question_body = question_row["Body"]
    question_title = question_row["Title"]
    question_score = question_row["Score"]
    formatted_data_dict[i] = {
                              "question_body": question_body,
                              "question_title": question_title,
                              "question_score": question_score,
                              "answers_and_scores": []}

    # Extracting answer(s) for the given question using ParentID of answer
    answers_for_question = answers_df[answers_df['ParentId'] == question_id]
    for j in range(len(answers_for_question)):
        answer_row = answers_for_question.iloc[j]
        answer_score = answer_row["Score"]
        answer_body = answer_row["Body"]
        formatted_data_dict[i]["answers_and_scores"].append([answer_body, answer_score])

    formatted_data_dict[i]["answers_and_scores"].sort(key=lambda x: x[1], reverse=True)
    
    tags_for_question = tags_df[tags_df["Id"] == question_id]
    formatted_data_dict[i]["tags"] = list(tags_for_question["Tag"])
    pprint(formatted_data_dict)
    break

  0%|          | 0/607282 [00:00<?, ?it/s]

{0: {'answers_and_scores': [["<p>Unfortunately the only API that isn't "
                             'deprecated is located in the ApplicationServices '
                             "framework, which doesn't have a bridge support "
                             "file, and thus isn't available in the bridge. If "
                             "you're wanting to use ctypes, you can use "
                             'ATSFontGetFileReference after looking up the '
                             'ATSFontRef.</p>\r\n'
                             '\r\n'
                             "<p>Cocoa doesn't have any native support, at "
                             'least as of 10.5, for getting the location of a '
                             'font.</p>',
                             12],
                            ['<p>open up a terminal '
                             '(Applications-&gt;Utilities-&gt;Terminal) and '
                             'type this in:</p>\r\n'
                             '

In [8]:
# 1. Summarize Answer stats per Question
answer_stats = answers_df.groupby('ParentId')['Score'].agg(['count', 'max', 'min', 'mean'])

# 2. Merge with Questions
# This gives you a single table with everything you need to make decisions
df_analysis = questions_df.merge(answer_stats, left_on='Id', right_index=True)

# 3. View the distribution
# This prints basic stats to help you pick a threshold
print(df_analysis[['Score', 'count', 'max', 'min']].describe())

# Filter: At least 2 answers
df_filtered = df_analysis[df_analysis['count'] >= 2].copy()

# Calculate the dynamic threshold (e.g., 50th percentile of 'max' scores)
score_threshold = df_filtered['max'].quantile(0.5) 

# Apply the filter
final_dataset = df_filtered[df_filtered['max'] >= score_threshold]

print(f"Questions remaining: {len(final_dataset)}")
print(f"Threshold set at: {score_threshold} upvotes")

# Now you have your 'Chosen' (the max) and your 'Rejected' (the min or second best)
# You can now iterate through this filtered list to build your DPO dict.

               Score          count            max            min
count  539238.000000  539238.000000  539238.000000  539238.000000
mean        2.504816       1.830587       3.988159       1.280991
std        20.452865       1.332178      27.129692       3.239398
min       -44.000000       1.000000      -7.000000     -38.000000
25%         0.000000       1.000000       1.000000       0.000000
50%         1.000000       1.000000       1.000000       1.000000
75%         2.000000       2.000000       3.000000       2.000000
max      5524.000000      55.000000    8384.000000     304.000000
Questions remaining: 161436
Threshold set at: 2.0 upvotes


In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from bs4 import BeautifulSoup
import re

# 1. Load Data
questions_df = pd.read_csv("../datasets/stackoverflow_dataset/Questions.csv", encoding='latin1')
answers_df = pd.read_csv("../datasets/stackoverflow_dataset/Answers.csv", encoding='latin1')
tags_df = pd.read_csv("../datasets/stackoverflow_dataset/Tags.csv", encoding='latin1')

def clean_html(html):
    """Converts SO HTML to clean text while preserving code blocks."""
    if not isinstance(html, str): return ""
    soup = BeautifulSoup(html, "html.parser")
    # Wrap code blocks in markdown backticks
    for code in soup.find_all('code'):
        code.replace_with(f"\n```\n{code.get_text()}\n```\n")
    return soup.get_text()

# 2. Optimized Grouping (O(N) instead of O(N^2))
print("Grouping tags and answers...")
tags_grouped = tags_df.groupby('Id')['Tag'].apply(lambda x: ", ".join(x.astype(str))).to_dict()

# We only need the top (chosen) and bottom (rejected) answers to save memory
answers_sorted = answers_df.sort_values(['ParentId', 'Score'], ascending=[True, False])
top_answers = answers_sorted.groupby('ParentId').head(1).set_index('ParentId')
bottom_answers = answers_sorted.groupby('ParentId').tail(1).set_index('ParentId')

# 3. Quality Analysis & Filtering
answer_stats = answers_df.groupby('ParentId')['Score'].agg(['count', 'max'])
df_merged = questions_df.merge(answer_stats, left_on='Id', right_index=True)

# Filter: Must have >= 2 answers AND Top answer must be in 90th percentile (Score >= 10-15 approx)
quality_cutoff = df_merged['max'].quantile(0.991)
df_final = df_merged[(df_merged['count'] >= 2) & (df_merged['max'] >= quality_cutoff)].copy()

print(f"Final dataset contains {len(df_final)} high-quality pairs.")

# 4. Constructing the RLHF Dataset
rlhf_dataset = []

for _, row in tqdm(df_final.iterrows(), total=len(df_final), desc="Building Pairs"):
    qid = row['Id']
    
    # Get answers
    chosen_row = top_answers.loc[qid]
    rejected_row = bottom_answers.loc[qid]
    
    # Skip if chosen and rejected are the same answer
    if chosen_row['Id'] == rejected_row['Id']:
        continue
        
    # Formatting Tags for the prompt
    tags = tags_grouped.get(qid, "general")
    prompt = f"Tags: {tags}\nTitle: {row['Title']}\nQuestion: {clean_html(row['Body'])}"
    
    # Scoring Logic: Log-Clipped Difference
    # S = ln(1 + score) to normalize power-law distribution
    s_c = np.log1p(max(0, chosen_row['Score']))
    s_r = np.log1p(max(0, rejected_row['Score']))
    
    # Weight represents the "confidence" of the preference for DPO/PPO

    weight = np.exp(s_c) / (np.exp(s_c) + np.exp(s_r))

    rlhf_dataset.append({
        "question_id": qid,
        "prompt": prompt,
        "chosen": clean_html(chosen_row['Body']),
        "rejected": clean_html(rejected_row['Body']),
        "chosen_score": float(s_c),
        "rejected_score": float(s_r),
        "preference_weight": float(weight)
    })

Grouping tags and answers...
Final dataset contains 4632 high-quality pairs.


Building Pairs: 100%|██████████| 4632/4632 [00:07<00:00, 650.86it/s]


In [3]:
final_rlhf_data_df = pd.DataFrame(rlhf_dataset)
final_rlhf_data_df.head()

,question_id,prompt,chosen,rejected,chosen_score,rejected_score,preference_weight
0,773,"Tags: python, iteration\nTitle: How do I use P...","As Sebastjan said, you first have to sort your...","Thanks for lots of nice anwser, this is a hack...",5.978886,1.098612,0.992462
1,972,"Tags: python, oop, methods, monkeypatching\nTi...","In Python, there is a difference between funct...","I don't know Python syntax, but I know Ruby ca...",6.357842,0.000000,0.998270
2,1476,"Tags: python, syntax, binary, integer, literal...",For reference—future Python possibilities:\nSt...,I am pretty sure this is one of the things due...,5.323010,0.000000,0.995146
3,1854,Tags: python\nTitle: How to check what OS am I...,\n```\n>>> import os\n>>> print os.name\nposix...,"Just for completeness, ""OS"" environment variab...",5.834811,0.000000,0.997085
4,2933,"Tags: python, user-interface, deployment, tkin...",First you will need some GUI library with Pyth...,You don't need to compile python for Mac/Windo...,5.247024,0.000000,0.994764


In [4]:
final_rlhf_data_df.to_csv("final_RLHF_stackoverflow_dataset_991perc.csv")

In [4]:
print(len(final_rlhf_data_df))

50546
